<a href="https://colab.research.google.com/github/SarahkhIT/DataEngProject/blob/main/notebooks/03_quality_gate_ge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Great Expectations Quality Gate

In [ ]:
import great_expectations as gx
import great_expectations.expectations as gxe

def run_paper_quality_checkpoint(df):
    context = gx.get_context()
    data_source = context.data_sources.add_pandas("papers")
    asset = data_source.add_dataframe_asset(name="papers_asset")
    batch = asset.build_batch_request(options={"dataframe": df})

    suite = gx.ExpectationSuite(name="paper_quality_suite")
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="title"))
    suite.add_expectation(gxe.ExpectColumnValueLengthsToBeBetween(column="abstract", min_value=1))
    suite.add_expectation(gxe.ExpectColumnValuesToMatchRegex(column="category", regex=r"^cs\."))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="published_date"))

    validator = context.get_validator(batch_request=batch, expectation_suite=suite)
    return validator.validate()

def run_ge_gate():
    result = run_paper_quality_checkpoint(bronze_df)
    if not result["success"]:
        raise RuntimeError("Quality gate failed — halting before Silver/Gold layer")
    print("✅ Quality gate passed — continuing to Silver/Gold")
    return result

run_with_lineage("quality_gate", run_ge_gate)

[OpenLineage] START — quality_gate
2026-07-22T22:51:22.744110Z [info     ] Could not find local file-backed GX project [great_expectations.data_context.data_context.context_factory] loc=context_factory.py:398
2026-07-22T22:51:22.746340Z [info     ] Created temporary directory '/tmp/tmplnjdcckj' for ephemeral docs site [great_expectations.data_context.types.base] loc=base.py:1384
2026-07-22T22:51:22.750056Z [info     ] Loading 'datasources' ->
[]    [great_expectations.datasource.fluent.config] loc=config.py:191
2026-07-22T22:51:22.801211Z [info     ] 	4 expectation(s) included in expectation_suite. [great_expectations.validator.validator] loc=validator.py:1035


Calculating Metrics:   0%|          | 0/20 [00:00<?, ?it/s]

✅ Quality gate passed — continuing to Silver/Gold
[OpenLineage] COMPLETE — quality_gate


{
  "success": true,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "papers-papers_asset",
          "column": "title"
        },
        "meta": {},
        "severity": "critical"
      },
      "result": {
        "element_count": 1500,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_value_lengths_to_be_between",
        "kwargs": {
          "batch_id": "papers-papers_asset",
          "column": "abstract",
          "min_value": 1
        },
        "meta": {},
        "severity": "critical"
      },
      "result": {
        "element_count": 150

### Proof: the gate correctly rejects bad data

In [ ]:
bad_ge_row = pd.DataFrame([{
    "paper_id": "X998",
    "title": None,                    # violates: title not null
    "abstract": "",                   # violates: abstract not empty
    "authors": "Nobody",
    "category": "INVALID.CAT",        # violates: doesn't match ^cs\.
    "published_date": None,           # violates: published_date not null
}])

test_df = pd.concat([bronze_df, bad_ge_row], ignore_index=True)
test_result = run_paper_quality_checkpoint(test_df)

print("Quality gate result on data WITH a bad row:", test_result["success"])
for r in test_result["results"]:
    if not r["success"]:
        print("❌ REJECTED:", r["expectation_config"]["type"],
              "| column:", r["expectation_config"]["kwargs"].get("column"),
              "| unexpected count:", r["result"].get("unexpected_count"))

2026-07-22T22:51:22.929366Z [info     ] Could not find local file-backed GX project [great_expectations.data_context.data_context.context_factory] loc=context_factory.py:398
2026-07-22T22:51:22.931552Z [info     ] Created temporary directory '/tmp/tmpjfhu_qcw' for ephemeral docs site [great_expectations.data_context.types.base] loc=base.py:1384
2026-07-22T22:51:22.934062Z [info     ] Loading 'datasources' ->
[]    [great_expectations.datasource.fluent.config] loc=config.py:191
2026-07-22T22:51:22.992684Z [info     ] 	4 expectation(s) included in expectation_suite. [great_expectations.validator.validator] loc=validator.py:1035


Calculating Metrics:   0%|          | 0/20 [00:00<?, ?it/s]

Quality gate result on data WITH a bad row: False
❌ REJECTED: expect_column_values_to_not_be_null | column: title | unexpected count: 1
❌ REJECTED: expect_column_value_lengths_to_be_between | column: abstract | unexpected count: 1
❌ REJECTED: expect_column_values_to_match_regex | column: category | unexpected count: 1
❌ REJECTED: expect_column_values_to_not_be_null | column: published_date | unexpected count: 1
